# Data collection and cleaning
Collects data from Arxiv, does basic data cleaning, and uploads it to supabase.

Requirements:
requests
feedparser
tqdm
supabase

In [12]:
import requests
import feedparser
from datetime import datetime, timedelta
import tarfile, io, os, re, time
import shutil
import tempfile
from tqdm import tqdm
from supabase import create_client, Client
import pandas as pd
from pprint import pp


In [ ]:
# base url for the arxiv api. 
BASE_URL = "http://export.arxiv.org/api/query"

# Do not change these parameters. Arxiv specifically asks for a 3 second delay between API calls
ARXIV_MAX_PER_REQUEST = 1000
SLEEP_TIME = 3

# These parameters can be changed as needed.
CATEGORIES = ["cs.DS", "cs.IT", "cs.CC", "math.co"]
DAYS_BACK = 7
KEYWORDS = [
      "locally+decodable+code",
      "matrix+concentration",
      "coding+theory",
      "hypergraph",
      "random+tensor",
      "matching+vectors",
      "rainbow+cycle",
    ]

def fetch_arxiv_metadata(categories, max_results, days_back=DAYS_BACK, keywords=None):
    """fetches metadata of papers uploaded to Arxiv in the specified `categories` up to 
    `days_back` days ago, which contain any specified `keywords`. 
    """
    cutoff_date = datetime.now() - timedelta(days=days_back)
    all_papers = []

    query = '(' + " OR ".join(f"cat:{c}" for c in categories) + ')'
    if keywords:
        keywords = [f"abs:\"{kw}\"" for kw in keywords]
        query += f" AND (" + " OR ".join(keywords) + ')'

    # Paginated fetch from the arxiv api. Will never need more than
    # one iteration for nightly pipeline, but this allows the same code
    # to be reused for the intial fetch.
    start = 0
    while start < max_results:
        params = {
            "search_query": query,
            "start": start,
            "max_results": min(ARXIV_MAX_PER_REQUEST, max_results - start),
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
        response = requests.get(BASE_URL, params=params)
        print(response.request.path_url)
        feed = feedparser.parse(response.text)

        if not feed.entries:
            return all_papers

        for entry in feed.entries:
            published = datetime.strptime(entry.updated, "%Y-%m-%dT%H:%M:%SZ")

            if published < cutoff_date:
                return all_papers

            paper = {
                "title": entry.title.strip(),
                "authors": [a.name for a in entry.authors],
                "published": published.isoformat(),
                "abstract": entry.summary.strip(),
                "id": entry.id.split("/abs/")[-1],
                "url": entry.link,
                "arxiv_categories": entry.arxiv_primary_category
            }

            all_papers.append(paper)
        start += ARXIV_MAX_PER_REQUEST
        time.sleep(SLEEP_TIME)
    return all_papers


if __name__ == "__main__":
    MAX_RESULTS = 1000
    papers = fetch_arxiv_metadata(CATEGORIES, MAX_RESULTS, DAYS_BACK, keywords=None)
    print(f"Collected {len(papers)} papers")

    # Example: print first paper
    print(papers[0])


/api/query?search_query=%28cat%3Acs.DS+OR+cat%3Acs.IT+OR+cat%3Acs.CC+OR+cat%3Amath.co%29&start=0&max_results=1000&sortBy=submittedDate&sortOrder=descending
{'id': 'http://arxiv.org/abs/2607.06551v1',
 'guidislink': True,
 'link': 'https://arxiv.org/abs/2607.06551v1',
 'title': "Tight Staircase Bounds for Cyclic Subsets below Dirac's Threshold",
 'title_detail': {'type': 'text/plain',
                  'language': None,
                  'base': '',
                  'value': 'Tight Staircase Bounds for Cyclic Subsets below '
                           "Dirac's Threshold"},
 'updated': '2026-07-07T17:53:58Z',
 'updated_parsed': time.struct_time(tm_year=2026, tm_mon=7, tm_mday=7, tm_hour=17, tm_min=53, tm_sec=58, tm_wday=1, tm_yday=188, tm_isdst=0),
 'links': [{'href': 'https://arxiv.org/abs/2607.06551v1',
            'rel': 'alternate',
            'type': 'text/html'},
           {'href': 'https://arxiv.org/pdf/2607.06551v1',
            'rel': 'related',
            'type': 'applicati

IndexError: list index out of range

In [3]:
def get_intro_text(session, arxiv_id):
    # --- download source ---
    url = f"https://arxiv.org/e-print/{arxiv_id}"

    try:
        r = session.get(url, timeout=60)
        if r.status_code != 200:
            return ""

        with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as tar:
            for member in tar:
                if not member.isfile() or not member.name.endswith(".tex"):
                    continue

                f = tar.extractfile(member)
                if not f:
                    continue

                text = f.read().decode(errors="ignore")

                # Check for main document
                if "\\begin{document}" not in text:
                    continue

                # --- extract introduction ---
                m = re.search(
                    r"\\section\*?\{[^}]*[intro|Intro][^}]*\}(.*?)(?=\\section|\Z)",
                    text,
                    re.IGNORECASE | re.DOTALL
                )
                if not m:
                    return ""

                intro = m.group(1)

                # --- minimal LaTeX cleanup ---
                # Remove comments
                intro = re.sub(r"%.*", "", intro)

                # # Remove common commands (light cleanup)
                # intro = re.sub(r"\\[a-zA-Z]+\{.*?\}", "", intro)
                # intro = re.sub(r"\\[a-zA-Z]+", "", intro)
                # # 2. Replace all block math environments with a placeholder
                # intro = re.sub(r"\\[.*?\\]", " <MATH> ", intro, flags=re.DOTALL) # Display math (\[...\])
                # intro = re.sub(
                #     r"\\begin\{equation\}.*?\\end\{equation\}",
                #     " <MATH> ", intro, flags=re.DOTALL
                # )

                # # 3. Replace all inline math ($...$) with a placeholder
                # intro = re.sub(r"\$.*?\$", " <MATH> ", intro, flags=re.DOTALL)

                # # 4. Preserve content of commands with arguments (e.g., \textbf{content} -> content)
                # intro = re.sub(r"\\[a-zA-Z]+\*?\{([^}]*)\}", r"\1", intro)

                # # 5. Remove any remaining backslash-prefixed commands or artifacts (like \1, \item, \section etc.)
                # intro = re.sub(r"\\[^\\s]+", "", intro)

                # 6. Normalize whitespace
                intro = re.sub(r"\\s+", " ", intro)
                return intro.strip()
    except (tarfile.ReadError, OSError):
        return ""
    return ""